# Incremental TESSERA embedding evolution: w1 → w2 → w3 → w4

This notebook follows the **same individual 10 m pixels** through all four cumulative prefix windows. It requires each field to be fully published in every window and each selected pixel to have a finite 128-D embedding in every window.

One field-balanced PCA basis is fitted on w1–w3 only and reused for every panel; w4 is transformed out of sample. Separate PCA fits per window are deliberately avoided because rotating axes would create false motion. Primary drift measurements use the full normalized 128-D vectors, not the 2-D plot.

In [ ]:
from functools import reduce
from pathlib import Path
import os
import sys

from IPython import get_ipython
from IPython.display import display

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "plain_tessera_incremental" / "__init__.py").is_file()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Run this notebook from inside the SpectraJam checkout.")
notebook_dir = repo_root / "plain_tessera_incremental" / "notebooks"
for path in (repo_root, notebook_dir):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from _pixel_analysis import (
    balanced_sample,
    clean_window_rows,
    l2_features,
    load_embeddings,
    load_static,
    scan_window,
)

OUTPUT_DIR = Path(
    os.environ.get(
        "TESSERA_OUTPUT_DIR",
        "/mnt/noobjam/harvard_tessera_incremental_v2",
    )
)
WINDOWS = ("w1", "w2", "w3", "w4")
LABELS = [
    "Maize",
    "Bean and Maize",
    "Irish Potato",
    "Bean",
    "Rice",
    "Irish Potato and Maize",
]
MIXTURE_PARENTS = {
    "Bean and Maize": ("Bean", "Maize"),
    "Irish Potato and Maize": ("Irish Potato", "Maize"),
}
MAX_FIELDS_PER_LABEL = 48
MAX_PIXELS_PER_FIELD = 16
MAX_TRAJECTORY_PIXELS_PER_LABEL = 30
# Numerical support cutoffs on unit-normalized 128-D vectors, not biological thresholds.
MIN_DIRECTION_CHORD = 1e-4
MIN_NET_DISPLACEMENT = 1e-4
MIN_DIRECTION_COHERENCE = 0.05
RANDOM_SEED = 20260708

print("Output:", OUTPUT_DIR)

In [ ]:
static = load_static(OUTPUT_DIR)
window_rows = []
for window in WINDOWS:
    spec = static.window_specs[window]
    start = pd.Timestamp(str(spec["start"]))
    end_exclusive = pd.Timestamp(str(spec["end_exclusive"]))
    duration_days = int((end_exclusive - start).days)
    in_contract = duration_days <= 366
    window_rows.append(
        {
            "window_id": window,
            "start": start.date().isoformat(),
            "end_exclusive": end_exclusive.date().isoformat(),
            "duration_days": duration_days,
            "in_contract": in_contract,
            "note": (
                "within annual contract"
                if in_contract
                else "OOD sensitivity; repeated day-of-year values"
            ),
        }
    )
window_table = pd.DataFrame(window_rows).set_index("window_id")
display(window_table)

candidate_field_uids = frozenset(
    static.fields.loc[static.fields["landcover"].isin(LABELS), "field_uid"]
    .astype(str)
)
if not candidate_field_uids:
    raise RuntimeError("None of the requested labels exist in fields.parquet.")
scans = {
    window: scan_window(
        static, window, retained_field_uids=candidate_field_uids
    )
    for window in WINDOWS
}

common_tasks = reduce(
    set.intersection, (set(scan.task_fingerprints) for scan in scans.values())
)
for task_key in common_tasks:
    fingerprints = {scans[window].task_fingerprints[task_key] for window in WINDOWS}
    if len(fingerprints) != 1:
        raise RuntimeError(f"Task fingerprint changes across windows: {task_key}")

common_published_fields = candidate_field_uids.intersection(
    *(set(scan.published_fields) for scan in scans.values())
)
if not common_published_fields:
    raise RuntimeError("No fields are fully published in all four windows yet.")
clean_by_window = {
    window: clean_window_rows(static, scans[window], common_published_fields)
    for window in WINDOWS
}
pair_sets = {
    window: set(
        clean.rows[["field_uid", "pixel_id"]].itertuples(index=False, name=None)
    )
    for window, clean in clean_by_window.items()
}
common_pairs = reduce(set.intersection, pair_sets.values())
if not common_pairs:
    raise RuntimeError("No clean finite pixel embeddings are shared by w1–w4.")
common_pair_index = pd.MultiIndex.from_tuples(
    sorted(common_pairs), names=["field_uid", "pixel_id"]
)
base_rows = clean_by_window["w4"].rows.set_index(["field_uid", "pixel_id"])
base_rows = base_rows.loc[base_rows.index.intersection(common_pair_index)].reset_index()
selected_base, selection_summary = balanced_sample(
    base_rows,
    LABELS,
    max_fields_per_label=MAX_FIELDS_PER_LABEL,
    max_pixels_per_field=MAX_PIXELS_PER_FIELD,
    seed=RANDOM_SEED,
    min_fields_per_label=2,
)
selected_pairs = selected_base[["field_uid", "pixel_id"]].copy()

timeline_parts = []
for window in WINDOWS:
    selected = selected_pairs.merge(
        clean_by_window[window].rows,
        on=["field_uid", "pixel_id"],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    if not selected["_merge"].eq("both").all():
        raise RuntimeError(f"Selected matched pixels disappeared from {window}.")
    selected = selected.drop(columns="_merge")
    loaded = load_embeddings(static, scans[window], selected)
    loaded["window_id"] = window
    loaded["_feature"] = list(l2_features(loaded))
    timeline_parts.append(loaded)
timeline = pd.concat(timeline_parts, ignore_index=True)

expected_rows = len(selected_pairs) * len(WINDOWS)
if len(timeline) != expected_rows or timeline.duplicated(
    ["field_uid", "pixel_id", "window_id"]
).any():
    raise RuntimeError("The matched four-window timeline is incomplete or duplicated.")
timeline["window_order"] = timeline["window_id"].map(
    {window: index for index, window in enumerate(WINDOWS)}
).astype(int)
timeline = timeline.sort_values(
    ["field_uid", "pixel_id", "window_order"], kind="stable"
).reset_index(drop=True)
count_columns = [
    "s2_source_count", "s1_source_count",
    "s2_valid_count", "s1_valid_count",
    "s2_input_count", "s1_input_count",
]
count_deltas = timeline.groupby(["field_uid", "pixel_id"])[count_columns].diff()
if (count_deltas.dropna().to_numpy(dtype=np.float64) < 0).any():
    raise RuntimeError("A cumulative observation/input count decreases across windows.")

diagnostics = pd.concat(
    {window: clean_by_window[window].diagnostics for window in WINDOWS}, axis=1
)
display(diagnostics)
display(selection_summary)
print(
    f"Matched cohort: {len(selected_pairs):,} physical pixels from "
    f"{selected_pairs['field_uid'].nunique():,} fields × {len(WINDOWS)} windows."
)

In [ ]:
fit_rows = timeline[timeline["window_id"].isin(("w1", "w2", "w3"))].copy()
fit_x = np.stack(fit_rows["_feature"].to_numpy())
field_pixel_counts = selected_pairs["field_uid"].value_counts().to_dict()
field_labels = selected_base[["field_uid", "landcover"]].drop_duplicates()
fields_per_label = field_labels["landcover"].value_counts().to_dict()
fit_weights = np.array(
    [
        1.0 / (
            3
            * field_pixel_counts[field_uid]
            * fields_per_label[label]
        )
        for field_uid, label in fit_rows[["field_uid", "landcover"]].itertuples(
            index=False, name=None
        )
    ],
    dtype=np.float64,
)
fit_weights /= fit_weights.sum()
pca_mean = np.average(fit_x, axis=0, weights=fit_weights)
weighted_centered = (fit_x - pca_mean) * np.sqrt(fit_weights[:, None])
_, singular_values, right_vectors = np.linalg.svd(weighted_centered, full_matrices=False)
pca_components = right_vectors[:3].copy()
for component in pca_components:
    anchor = int(np.argmax(np.abs(component)))
    if component[anchor] < 0:
        component *= -1
explained_ratio = singular_values[:3] ** 2 / np.sum(singular_values**2)

all_x = np.stack(timeline["_feature"].to_numpy())
all_centered = all_x - pca_mean
all_scores = all_centered @ pca_components.T
reconstructed = all_scores @ pca_components
reconstruction_residual = np.linalg.norm(
    all_centered - reconstructed, axis=1
) / np.maximum(np.linalg.norm(all_centered, axis=1), 1e-12)
timeline["pca_1"] = all_scores[:, 0]
timeline["pca_2"] = all_scores[:, 1]
timeline["pca_3"] = all_scores[:, 2]
timeline["pca_reconstruction_residual"] = reconstruction_residual
timeline["raw_embedding_norm"] = [
    float(np.linalg.norm(vector)) for vector in timeline["_vector"]
]
display(
    pd.DataFrame(
        {"explained_variance_ratio": explained_ratio},
        index=["PC1", "PC2", "PC3"],
    )
)
display(
    timeline.groupby("window_id")["pca_reconstruction_residual"]
    .agg(["median", lambda values: values.quantile(0.9)])
    .rename(columns={"<lambda_0>": "p90"})
    .reindex(WINDOWS)
)

In [ ]:
palette = dict(zip(LABELS, plt.get_cmap("tab10").colors, strict=False))
x_limits = (timeline["pca_1"].quantile(0.005), timeline["pca_1"].quantile(0.995))
y_limits = (timeline["pca_2"].quantile(0.005), timeline["pca_2"].quantile(0.995))
figure, axes = plt.subplots(2, 2, figsize=(15, 12), sharex=True, sharey=True)
for axis, window in zip(axes.ravel(), WINDOWS, strict=True):
    window_rows = timeline[timeline["window_id"].eq(window)]
    for label in LABELS:
        rows = window_rows[window_rows["landcover"].eq(label)]
        axis.scatter(
            rows["pca_1"], rows["pca_2"], s=9, alpha=0.4, linewidths=0,
            color=palette[label], label=label,
        )
    suffix = " — OOD sensitivity" if window == "w4" else ""
    axis.set_title(f"{window}{suffix}: {len(window_rows):,} matched pixels")
    axis.set_xlim(*x_limits)
    axis.set_ylim(*y_limits)
    axis.grid(alpha=0.12)
    axis.set_xlabel("fixed PC1")
    axis.set_ylabel("fixed PC2")
handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, bbox_to_anchor=(1.01, 0.9), loc="upper left")
figure.suptitle(
    "Same physical pixels in one fixed w1–w3 PCA basis (w4 projected out of sample)",
    fontsize=14,
)
figure.tight_layout()
plt.show()

In [ ]:
trajectory_rng = np.random.default_rng(RANDOM_SEED + 1)
trajectory_pixel_ids = []
for label in LABELS:
    candidates = np.array(
        sorted(timeline.loc[timeline["landcover"].eq(label), "pixel_id"].unique()),
        dtype=object,
    )
    if len(candidates) > MAX_TRAJECTORY_PIXELS_PER_LABEL:
        candidates = trajectory_rng.choice(
            candidates, size=MAX_TRAJECTORY_PIXELS_PER_LABEL, replace=False
        )
    trajectory_pixel_ids.extend(candidates.tolist())
trajectory_rows = timeline[timeline["pixel_id"].isin(trajectory_pixel_ids)]

figure, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True, sharey=True)
for axis, label in zip(axes.ravel(), LABELS, strict=True):
    rows = trajectory_rows[trajectory_rows["landcover"].eq(label)]
    for _, pixel_rows in rows.groupby("pixel_id", sort=False):
        pixel_rows = pixel_rows.sort_values("window_order")
        axis.plot(
            pixel_rows["pca_1"], pixel_rows["pca_2"],
            color=palette[label], alpha=0.22, linewidth=0.8,
        )
        axis.scatter(
            pixel_rows["pca_1"], pixel_rows["pca_2"],
            c=pixel_rows["window_order"], cmap="viridis", vmin=0, vmax=3,
            s=12, alpha=0.7,
        )
    axis.set_title(f"{label}: {rows['pixel_id'].nunique()} pixel trajectories")
    axis.grid(alpha=0.12)
    axis.set_xlabel("fixed PC1")
    axis.set_ylabel("fixed PC2")
color_map = plt.cm.ScalarMappable(
    norm=plt.Normalize(vmin=0, vmax=3), cmap="viridis"
)
color_bar = figure.colorbar(
    color_map, ax=axes.ravel().tolist(), ticks=range(4), fraction=0.02, pad=0.02
)
color_bar.ax.set_yticklabels(WINDOWS)
color_bar.set_label("cumulative window")
figure.suptitle("Individual matched-pixel trajectories; marker color runs w1→w4")
figure.tight_layout()
plt.show()

field_locations = timeline.groupby(
    ["field_uid", "landcover", "window_id", "window_order"], as_index=False
)[["pca_1", "pca_2"]].mean()
crop_locations = field_locations.groupby(
    ["landcover", "window_id", "window_order"], as_index=False
)[["pca_1", "pca_2"]].mean()
figure, axis = plt.subplots(figsize=(10, 8))
for label in LABELS:
    rows = crop_locations[crop_locations["landcover"].eq(label)].sort_values("window_order")
    axis.plot(rows["pca_1"], rows["pca_2"], marker="o", color=palette[label], label=label)
    for _, row in rows.iterrows():
        axis.annotate(row["window_id"], (row["pca_1"], row["pca_2"]), fontsize=8)
axis.set_xlabel("fixed PC1")
axis.set_ylabel("fixed PC2")
axis.set_title("Field-balanced crop trajectory summaries (individual trajectories are above)")
axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
axis.grid(alpha=0.15)
figure.tight_layout()
plt.show()

In [ ]:
window_frames = {
    window: timeline[timeline["window_id"].eq(window)].set_index("pixel_id")
    for window in WINDOWS
}
transition_records = []
for start, end in zip(WINDOWS[:-1], WINDOWS[1:], strict=True):
    before = window_frames[start].sort_index()
    after = window_frames[end].reindex(before.index)
    before_x = np.stack(before["_feature"].to_numpy())
    after_x = np.stack(after["_feature"].to_numpy())
    delta = after_x - before_x
    chord = np.linalg.norm(delta, axis=1)
    cosine_drift = 1.0 - np.sum(before_x * after_x, axis=1)
    for position, pixel_id in enumerate(before.index):
        transition_records.append(
            {
                "pixel_id": pixel_id,
                "field_uid": before.iloc[position]["field_uid"],
                "landcover": before.iloc[position]["landcover"],
                "transition": f"{start}→{end}",
                "cosine_drift": cosine_drift[position],
                "chord_distance": chord[position],
                "log_norm_change": np.log(
                    max(float(after.iloc[position]["raw_embedding_norm"]), 1e-12)
                )
                - np.log(max(float(before.iloc[position]["raw_embedding_norm"]), 1e-12)),
                "s2_valid_added": int(after.iloc[position]["s2_valid_count"] - before.iloc[position]["s2_valid_count"]),
                "s1_valid_added": int(after.iloc[position]["s1_valid_count"] - before.iloc[position]["s1_valid_count"]),
                "s2_input_added": int(after.iloc[position]["s2_input_count"] - before.iloc[position]["s2_input_count"]),
                "s1_input_added": int(after.iloc[position]["s1_input_count"] - before.iloc[position]["s1_input_count"]),
                "direction_supported": bool(chord[position] >= MIN_DIRECTION_CHORD),
                "_delta_direction": (
                    delta[position] / chord[position]
                    if chord[position] >= MIN_DIRECTION_CHORD
                    else None
                ),
            }
        )
drift = pd.DataFrame(transition_records)
field_drift = drift.groupby(
    ["field_uid", "landcover", "transition"], as_index=False
).agg(
    median_cosine_drift=("cosine_drift", "median"),
    p90_cosine_drift=("cosine_drift", lambda values: values.quantile(0.9)),
    median_chord_distance=("chord_distance", "median"),
    median_s2_valid_added=("s2_valid_added", "median"),
    median_s1_valid_added=("s1_valid_added", "median"),
    median_s2_input_added=("s2_input_added", "median"),
    median_s1_input_added=("s1_input_added", "median"),
)
transition_order = [f"{start}→{end}" for start, end in zip(WINDOWS[:-1], WINDOWS[1:], strict=True)]
drift_heatmap = field_drift.groupby(["landcover", "transition"])["median_cosine_drift"].median().unstack().reindex(index=LABELS, columns=transition_order)

figure, axes = plt.subplots(1, 2, figsize=(16, 6))
image = axes[0].imshow(drift_heatmap.to_numpy(), cmap="magma", aspect="auto")
axes[0].set_xticks(range(len(transition_order)), transition_order)
axes[0].set_yticks(range(len(LABELS)), LABELS)
axes[0].set_title("Median field-level 128-D cosine drift")
for row in range(len(LABELS)):
    for column in range(len(transition_order)):
        axes[0].text(column, row, f"{drift_heatmap.iloc[row, column]:.3f}", ha="center", va="center", color="white")
figure.colorbar(image, ax=axes[0], fraction=0.046, pad=0.04)

box_data = []
box_labels = []
for transition in transition_order:
    for label in LABELS:
        values = field_drift.loc[
            field_drift["transition"].eq(transition)
            & field_drift["landcover"].eq(label),
            "median_cosine_drift",
        ].to_numpy()
        box_data.append(values)
        box_labels.append(f"{transition}\n{label}")
axes[1].boxplot(box_data, showfliers=False)
axes[1].set_xticks(range(1, len(box_labels) + 1), box_labels, rotation=75, ha="right", fontsize=7)
axes[1].set_ylabel("field median cosine drift")
axes[1].set_title("Between-field variation (pixels summarized within field)")
axes[1].grid(axis="y", alpha=0.15)
figure.tight_layout()
plt.show()

display(
    field_drift.groupby(["landcover", "transition"])[
        [
            "median_cosine_drift", "median_chord_distance",
            "median_s2_valid_added", "median_s1_valid_added",
            "median_s2_input_added", "median_s1_input_added",
        ]
    ].median().reindex(pd.MultiIndex.from_product([LABELS, transition_order], names=["landcover", "transition"]))
)

In [ ]:
path_records = []
for (field_uid, pixel_id), rows in timeline.groupby(["field_uid", "pixel_id"], sort=False):
    rows = rows.sort_values("window_order")
    vectors = np.stack(rows["_feature"].to_numpy())
    segment_lengths = np.linalg.norm(np.diff(vectors, axis=0), axis=1)
    path_length = float(segment_lengths.sum())
    net_displacement = float(np.linalg.norm(vectors[-1] - vectors[0]))
    path_records.append(
        {
            "field_uid": field_uid,
            "pixel_id": pixel_id,
            "landcover": rows.iloc[0]["landcover"],
            "path_length": path_length,
            "net_displacement": net_displacement,
            "net_supported": bool(net_displacement >= MIN_NET_DISPLACEMENT),
            "tortuosity": (
                path_length / net_displacement
                if net_displacement >= MIN_NET_DISPLACEMENT
                else np.nan
            ),
        }
    )
paths = pd.DataFrame(path_records)
field_paths = paths.groupby(["field_uid", "landcover"], as_index=False).agg(
    median_path_length=("path_length", "median"),
    median_net_displacement=("net_displacement", "median"),
    median_tortuosity=("tortuosity", "median"),
)
path_support = paths.groupby("landcover").agg(
    total_pixels=("pixel_id", "size"),
    effective_net_pixels=("net_supported", "sum"),
)
path_support["below_net_threshold_pixels"] = (
    path_support["total_pixels"] - path_support["effective_net_pixels"]
)
display(
    field_paths.groupby("landcover")[
        ["median_path_length", "median_net_displacement", "median_tortuosity"]
    ].median().reindex(LABELS)
)
display(path_support.reindex(LABELS))

direction_records = []
for transition in transition_order:
    transition_rows = drift[drift["transition"].eq(transition)]
    for label in LABELS:
        rows = transition_rows[transition_rows["landcover"].eq(label)]
        effective = rows[rows["direction_supported"]]
        effective_count = len(effective)
        stationary_count = len(rows) - effective_count
        if effective_count:
            pixel_counts = effective["field_uid"].value_counts().to_dict()
            weights = np.array(
                [1.0 / pixel_counts[field_uid] for field_uid in effective["field_uid"]]
            )
            direction_matrix = np.stack(effective["_delta_direction"].to_numpy())
            mean_direction = np.average(direction_matrix, axis=0, weights=weights)
            coherence = float(np.linalg.norm(mean_direction))
        else:
            mean_direction = None
            coherence = np.nan
        direction_supported = bool(
            effective_count and coherence >= MIN_DIRECTION_COHERENCE
        )
        direction_records.append(
            {
                "landcover": label,
                "transition": transition,
                "total_pixels": len(rows),
                "effective_direction_pixels": effective_count,
                "stationary_pixels": stationary_count,
                "effective_direction_fields": effective["field_uid"].nunique(),
                "direction_coherence": coherence,
                "direction_supported": direction_supported,
                "_mean_direction": (
                    mean_direction / coherence
                    if direction_supported
                    else np.full(len(timeline.iloc[0]["_feature"]), np.nan, dtype=np.float64)
                ),
            }
        )
directions = pd.DataFrame(direction_records)
similarity_records = []
for mixture, parents in MIXTURE_PARENTS.items():
    for transition in transition_order:
        mixture_direction = directions.loc[
            directions["landcover"].eq(mixture) & directions["transition"].eq(transition),
            "_mean_direction",
        ].iloc[0]
        record = {
            "mixture": mixture,
            "transition": transition,
            "parent_a": parents[0],
            "parent_b": parents[1],
        }
        for parent_index, parent in enumerate(parents):
            parent_direction = directions.loc[
                directions["landcover"].eq(parent) & directions["transition"].eq(transition),
                "_mean_direction",
            ].iloc[0]
            similarity = (
                float(np.dot(mixture_direction, parent_direction))
                if (
                    np.isfinite(mixture_direction).all()
                    and np.isfinite(parent_direction).all()
                )
                else np.nan
            )
            record[f"similarity_parent_{'a' if parent_index == 0 else 'b'}"] = similarity
        similarity_records.append(record)
direction_similarity = pd.DataFrame(similarity_records)
direction_report_index = pd.MultiIndex.from_product(
    [LABELS, transition_order], names=["landcover", "transition"]
)
display(
    directions.drop(columns="_mean_direction")
    .set_index(["landcover", "transition"])
    .reindex(direction_report_index)
)
display(direction_similarity.set_index(["mixture", "transition"]))

## Interpretation

- Every plotted line is the same physical pixel followed through w1–w4. Field/crop lines average only plotted coordinates or scalar drift summaries; no field embedding is created.
- All windows are cumulative prefixes beginning 2024-09-01. `w2 − w1` is representation displacement after recomputing the longer sequence, not an embedding of only the newly added dates.
- Cosine drift and chord distance use the full normalized 128-D vectors. PCA is a fixed low-dimensional diagnostic and may hide out-of-plane motion; reconstruction residual reports that limitation.
- Added observations and changes in TESSERA's 8…256 input buckets can move embeddings even without biological change. Read drift alongside S1/S2 valid and input-count increments.
- Direction uses only adjacent movements with normalized-vector chord distance at least `MIN_DIRECTION_CHORD=1e-4`; smaller movements are reported as stationary and are excluded from direction estimates. Tortuosity is `NaN` when net displacement is below `MIN_NET_DISPLACEMENT=1e-4`.
- Direction coherence near 1 means supported pixel movements align. If no movement is supported or coherence is below `MIN_DIRECTION_COHERENCE=0.05`, the unit direction and parent similarities are reported as `NaN` rather than interpreted.
- These fixed cutoffs are numerical support/interpretability rules on unit-normalized 128-D vectors, not biological thresholds. Check sensitivity to nearby values before making a scientific claim.
- `w4` spans 487 days, beyond TESSERA v1.1's annual training contract. Day-of-year repeats without a year token. Treat w1–w3 as the primary in-contract evolution and w4 only as an out-of-distribution sensitivity experiment.
- These field-grouped summaries are not spatially blocked. Use fixed geographic blocks and independent regions before interpreting differences as general crop-growth signatures.